In [1]:
from typing import Optional, Callable, Any
import datetime
from tqdm import tqdm
import polars as pl
import torch
import torch_geometric as tg

# Example: Using the C19EpidemiologyGraphs database to create a Torch Geometric dataset for Graph Neural Network (GNN) usage

This example notebook serves to show an use case of the C19EpidemiologyGraphs database. Specifically, we are first processing the data to fit it into what we need (rolling time windows + interpolation) and then we are creating a Torch Geometric dataset, which is a frequent step before a Graph Neural Network can be trained. Detailed explanations regarding Torch Geometric's dataset creation process are beyond our scope, but we refer the curious reader to the official tutorial on the matter: https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html.
It's worth mentioning that the linked tutorial tackles the creation of in-memory datasets, and here we are going to use an on-disk dataset, where the entries are stored in an SQLite database (a completely different one than C19EpidemiologyGraphs).

For this dataset, we are going to use sliding time windows of 7, 14 and 30 days on the epidemiology table. We are using the columns related to testing, and as they contain a large proportion of null values, we are also going to apply some interpolation. However, we don't want most of the data to be interpolated, so we're gonna set the following filter that every window needs to pass in order to not be discarded:
- Relevant columns must have at *least* 90% of its days present (interpolated or not); and
- Relevant columns must have at *least* 80% of its days present specifically as non-interpolated values.
This processing was done by reading the full table into memory and then using Polars to do everything. It could be possible to apply a user-defined function to SQLite by using its foreign function interface and then query it, as to not have to load the entire table into memory, but we think this would add another layer of complicatedness into this, specially given the table *can* reasonably fit into RAM.

Different use cases would warrant different processing steps or filters, and this is only showing one possible set of restrictions and transformations. All of this is to show the flexibility of C19EpidemiologyGraphs.

In [10]:
# Several columns from epidemiology and vertices are going to be ignored in this example,
#   such as all metrics related to recovered patients.
sqlite_filename = "C19EpidemiologyGraphs.sqlite3"
uri = "sqlite://" + sqlite_filename
query_valid_epidemiology = """
    SELECT v.aggregation_level, e.date, e.location_key,
           e.new_confirmed, e.new_deceased, e.new_tested,
           e.cumulative_confirmed, e.cumulative_deceased, e.cumulative_tested,
           v.ROWID AS node_rowid, v.x_longitude, v.y_latitude,
           CASE
               WHEN v.aggregation_level = 2 THEN v.country_name
               ELSE ''
           END AS grouped_by_country
    FROM epidemiology AS e
    INNER JOIN vertices AS v ON e.location_key = v.location_key
    WHERE e.new_confirmed IS NOT NULL AND e.cumulative_confirmed IS NOT NULL
"""

epidemiology = pl.read_database_uri(
    query=query_valid_epidemiology, uri=uri, engine="connectorx",
    partition_on="date", partition_num=10, partition_range=(18261, 19356)
).with_columns(
    date=pl.col("date").cast(pl.Date)
)

epidemiology.head()

aggregation_level,date,location_key,new_confirmed,new_deceased,new_tested,cumulative_confirmed,cumulative_deceased,cumulative_tested,node_rowid,x_longitude,y_latitude,grouped_by_country
i64,date,str,i64,i64,i64,i64,i64,i64,i64,f64,f64,str
0,2020-01-01,"""AD""",0,0,null,0,0,null,204,1.555278,42.558334,""""""
0,2020-01-01,"""AE""",0,0,null,0,0,null,13603,54.299999,24.4,""""""
0,2020-01-01,"""AI""",0,0,null,0,0,null,9803,-63.048889,18.227222,""""""
0,2020-01-01,"""AL""",0,0,null,0,0,null,11668,20.0,41.0,""""""
0,2020-01-01,"""AG""",0,0,null,0,0,null,13604,-61.849998,17.116667,""""""


In [11]:
# Apply the interpolation. Here we are using 2 types: number of deaths get a regular linear interpolation, but as the number of confirmed cases and the number of tests at a given day are related, we can interpolated the tests column by what is present in the confirmed column first, and then interpolate normally when that fails due to division by zero.
epidemiology = epidemiology.with_columns(
    new_deceased_interpolated=pl.col("new_deceased").interpolate().over(
        partition_by="location_key", order_by="date"
    ).round(decimals=0, mode="half_away_from_zero").cast(pl.Int32),
    new_tested_interpolated=pl.col("new_tested").interpolate_by("new_confirmed").over(
        partition_by="location_key", order_by="date"
    ).fill_nan(None).interpolate().over(
        partition_by="location_key", order_by="date"
    ).round(decimals=0, mode="half_away_from_zero").cast(pl.Int32),
)

# Then, we use apply the rolling time windows of 7, 14 and 30 days.
# We aggregate them by addition along the new counts and by picking up the last non-null element for the cumulative counts.
window_sizes = [7, 14, 30]
def apply_window(df, window_size):
    return df.rolling(
        index_column="date", period=str(window_size) + "d",
        group_by="location_key"
    ).agg(
        new_confirmed=pl.when(
            pl.col("new_confirmed").count() > 0
        ).then(pl.col("new_confirmed").sum()),
        new_deceased=pl.when(
            pl.col("new_deceased").count() > 0
        ).then(pl.col("new_deceased").sum()),
        new_tested=pl.when(
            pl.col("new_tested").count() > 0
        ).then(pl.col("new_tested").sum()),
        new_deceased_interpolated=pl.when(
            pl.col("new_deceased_interpolated").count() > 0
        ).then(pl.col("new_deceased_interpolated").sum()),
        new_tested_interpolated=pl.when(
            pl.col("new_tested_interpolated").count() > 0
        ).then(pl.col("new_tested_interpolated").sum()),
        cumulative_confirmed=pl.col("cumulative_confirmed").last(ignore_nulls=True),
        cumulative_deceased=pl.col("cumulative_deceased").last(ignore_nulls=True),
        cumulative_tested=pl.col("cumulative_tested").last(ignore_nulls=True),
        num_non_nulls_in_real_confirmed=pl.col("new_confirmed").is_not_null().sum(),
        num_non_nulls_in_real_deceased=pl.col("new_deceased").is_not_null().sum(),
        num_non_nulls_in_real_tested=pl.col("new_tested").is_not_null().sum(),
        num_non_nulls_in_interpolated_deceased=pl.col("new_deceased_interpolated").is_not_null().sum(),
        num_non_nulls_in_interpolated_tested=pl.col("new_tested_interpolated").is_not_null().sum(),
        window_size=window_size,
        grouped_by_country=pl.col("grouped_by_country").last(),
        aggregation_level=pl.col("aggregation_level").last(),
        node_rowid=pl.col("node_rowid").last(),
        x_longitude=pl.col("x_longitude").last(),
        y_latitude=pl.col("y_latitude").last()
    )
epidemiology = epidemiology.sort(by="date")
epidemiology = pl.concat([apply_window(epidemiology, w) for w in window_sizes])

# And finally, we apply the filter and select only the relevant columns
min_non_interpolated = 0.8
min_non_absent = 0.9
epidemiology = epidemiology.filter(
    # At least 80% of values present AND non-interpolated
    pl.col("num_non_nulls_in_real_deceased") / pl.col("window_size") >= min_non_interpolated,
    pl.col("num_non_nulls_in_real_tested") / pl.col("window_size") >= min_non_interpolated,

    # At least 90% of values present, real or interpolated
    pl.col("num_non_nulls_in_real_confirmed") / pl.col("window_size") >= min_non_absent,
    pl.col("num_non_nulls_in_interpolated_deceased") / pl.col("window_size") >= min_non_absent,
    pl.col("num_non_nulls_in_interpolated_tested") / pl.col("window_size") >= min_non_absent,

    # Values must all be non-negative
    pl.col("new_confirmed") >= 0,
    pl.col("new_deceased_interpolated") >= 0,
    pl.col("new_tested_interpolated") >= 0,

    # Number of tests performed must be greater than or equal to the number of confirmed cases
    pl.col("new_tested_interpolated") >= pl.col("new_confirmed")
).select(
    "aggregation_level", pl.col("date").alias("window_final_day"), "window_size", "location_key",
    "new_confirmed", "new_deceased_interpolated", "new_tested_interpolated",
    "cumulative_confirmed", "cumulative_deceased", "cumulative_tested",
    "node_rowid", "x_longitude", "y_latitude", "grouped_by_country"
)

epidemiology.head()

aggregation_level,window_final_day,window_size,location_key,new_confirmed,new_deceased_interpolated,new_tested_interpolated,cumulative_confirmed,cumulative_deceased,cumulative_tested,node_rowid,x_longitude,y_latitude,grouped_by_country
i64,date,i32,str,i64,i32,i64,i64,i64,i64,i64,f64,f64,str
0,2020-03-28,7,"""AE""",372,1,85782,570,3,198848,13603,54.299999,24.4,""""""
0,2020-03-29,7,"""AE""",416,4,76728,664,6,206542,13603,54.299999,24.4,""""""
0,2020-03-30,7,"""AE""",331,4,81093,664,6,220000,13603,54.299999,24.4,""""""
0,2020-03-31,7,"""AE""",481,6,92037,814,8,240480,13603,54.299999,24.4,""""""
0,2020-04-01,7,"""AE""",691,6,100261,1024,8,262913,13603,54.299999,24.4,""""""


The filtered epidemiology table is then grouped by the location's aggregation level, window final date, window size and, sometimes, country name. The country name is only added as a group column if, and only if, aggregation level is 2. Then, for each group, we obtain the graph nodes and edges, based on whether the locations involved have epidemiological information in the current group. This creates a graph, which is then broken into it's connected components. Lastly, only components with size (measured as the number of nodes) larger than or equal to 8 get added to the dataset.

In [19]:
class ExampleDatasetClass(tg.data.OnDiskDataset):
    def __init__(
            self,
            root: str,
            epidemiology_df : pl.DataFrame,
            min_graph_size: int = 8,
            transform: Optional[Callable] = None,
            pre_transform: Optional[Callable] = None,
            pre_filter: Optional[Callable] = None,
            backend: str = "sqlite",
            # Minimum number of torch geometric graphs accumulated in memory before a SQLite insert.
            block_size: int = 4096,
            log: bool = True
    ) -> None:
        self.epidemiology_df = epidemiology_df
        self.min_graph_size = min_graph_size
        self.pre_transform = pre_transform
        self.block_size = block_size
        schema = {
            "x": dict(dtype=torch.int32, size=(-1, 1)),
            "edge_index": dict(dtype=torch.int64, size=(2, -1)),
            "edge_attr": dict(dtype=torch.float32, size=(-1, 1)),
            "pos": dict(dtype=torch.float32, size=(-1, 2)),
            "new_epidemiology": dict(dtype=torch.int32, size=(-1, 3)),
            "cumulative_epidemiology": dict(dtype=torch.int64, size=(-1, 3)),
            "aggregation_level": int,
            "window_final_day": int,
            "window_size": int,
            "grouped_by_country": str,
            "component_index": int
        }
        super().__init__(root, transform, pre_filter, backend, schema, log)

    @property
    def raw_file_names(self):
        return [sqlite_filename]

    def process_and_extend(self, data_list):
        if self.pre_filter is not None:
            data_list = filter(self.pre_filter, data_list)
        if self.pre_transform is not None:
            data_list = map(self.pre_transform, data_list)
        if self.pre_filter is not None or self.pre_transform is not None:
            data_list = list(data_list)
        if len(data_list) > 0:
            self.extend(data_list)

    def process(self):
        uri = "sqlite://" + self.root + "/" + self.raw_file_names[0]
        group_columns = ["aggregation_level", "window_final_day", "window_size", "grouped_by_country"]
        epidemiology_groups = self.epidemiology_df.group_by(group_columns)
        num_epidemiology_groups = self.epidemiology_df.n_unique(subset=group_columns)
        data_list = []
        for ((aggregation_level, window_final_day, window_size, grouped_by_country),
             group) in tqdm(epidemiology_groups, total=num_epidemiology_groups):
            group_location_keys = group.get_column("location_key").to_list()
            nodes = group.select("node_rowid", "x_longitude", "y_latitude",
                                 "new_confirmed", "new_deceased_interpolated", "new_tested_interpolated",
                                 "cumulative_confirmed", "cumulative_deceased", "cumulative_tested")
            # A graph's edges are stored as indices relative to the position of the nodes in graph.x, so we need
            #   to translate the fixed and unique location_key/rowid (in this case, rowid) of a vertex
            #   into a relative index that's starting from 0 and is monotonically increasing with
            #   every subsequent node.
            # This is done by using polars' DataFrame.with_row_index() to create a 1-to-1 mapping that can then
            #   be joined with the edges' source nodes, and again with the destination nodes.
            nodes_map = nodes.with_row_index().select("node_rowid", "index")
            q_edges = """
                WITH curr_nodes AS (SELECT location_key, ROWID AS node_rowid
                                    FROM vertices
                                    WHERE location_key IN ({0}))
                SELECT source_verts.node_rowid AS source_id,
                       destination_verts.node_rowid AS destination_id,
                       edges.distance_m
                FROM edges
                INNER JOIN curr_nodes AS source_verts ON edges.source = source_verts.location_key
                INNER JOIN curr_nodes AS destination_verts ON edges.destination = destination_verts.location_key
            """.format(', '.join(['?'] * len(group)))
            edges = pl.read_database_uri(query=q_edges, uri=uri, engine="adbc",
                                         execute_options={"parameters": tuple(group_location_keys)}).join(
                nodes_map, how="inner", left_on="source_id", right_on="node_rowid", maintain_order="left"
            ).join(
                nodes_map, how="inner", left_on="destination_id", right_on="node_rowid",
                suffix="_destination", maintain_order="left"
            )

            # With all the required information related to the graph and with its nodes and edges collected,
            #   it's possible to instantiate a Torch Geometric graph.
            # We need to convert Polars' DataFrames to Torch tensors,
            #   but 0-dimensional ints and strings can remain as primitives.
            graph = tg.data.Data(
                # The x feature is mandatory in Torch Geometric. Here, we used the node's unique rowid in the database
                x=nodes.select("node_rowid").to_torch(dtype=pl.Int32),
                edge_index=edges.select("index", "index_destination").to_torch(dtype=pl.Int64).T,
                edge_attr=edges.select("distance_m").to_torch(dtype=pl.Float32),
                pos=nodes.select("x_longitude", "y_latitude").to_torch(dtype=pl.Float32),
                new_epidemiology=nodes.select("new_confirmed", "new_deceased_interpolated",
                                              "new_tested_interpolated").to_torch(dtype=pl.Int32),
                cumulative_epidemiology=nodes.select("cumulative_confirmed", "cumulative_deceased",
                                                     "cumulative_tested").to_torch(dtype=pl.Int64),
                aggregation_level=aggregation_level,
                # Like in C19EpidemiologyGraphs, dates in this dataset's SQLite database are stored as the
                #   number of days since the Unix epoch (01/01/1970).
                #   However, when the database entries are materialized as actual Torch Geometric graphs
                #   (like here, before this graph is written into the database),
                #   it's possible to use datetime objects, to better represent these dates.
                # The conversion to and from datetime and the number of days since the Unix epoch can be seen
                #   in the methods serialize and deserialize, a bit under.
                window_final_day=window_final_day,
                window_size=window_size,
                grouped_by_country=grouped_by_country
            ).coalesce()

            # For this example, we're not adding this full graph, but rather it's connected components.
            #   Torch Geometric's Data objects have a method to get a graph's connected components.
            # We also take this moment to filter these components based on their size, measured as number of nodes.
            graph_components = list(filter(lambda data: data.num_nodes >= self.min_graph_size,
                                           graph.connected_components()))
            for i in range(len(graph_components)):
                graph_components[i].component_index = i
            data_list.extend(graph_components)

            # Note that the length of data_list usually exceeds block_size a bit,
            #   as this check is outside the group for-loop, therefore happening only once per
            #   C19EpidemiologyGraphs' graph entry, which can include many entries into the dataset.
            # No reason in particular for this to be done like that,
            #   just a very small memory-performance tradeoff decision.
            if len(data_list) >= self.block_size:
                self.process_and_extend(data_list)
                data_list = []

        self.process_and_extend(data_list)

    # Convert the window_starting_date from a datetime object to the number of days since the Unix epoch.
    def serialize(self, data: tg.data.data.BaseData) -> Any:
        d = data.to_dict()
        d["window_final_day"] = (d["window_final_day"] - datetime.date(1970, 1, 1)).days
        return d

    # Convert the window_starting_date from the number of days since the Unix epoch to a datetime object.
    def deserialize(self, data: Any) -> tg.data.data.BaseData:
        data["window_final_day"] = datetime.date(1970, 1, 1) + datetime.timedelta(days=data["window_final_day"])
        return tg.data.Data.from_dict(data)

In [20]:
# I don't think it's best practice to pass the epidemiology dataframe directly,
#   and it'd be better to save it to a file and pass its name instead, but for our purposes, this suffices.
ds = ExampleDatasetClass(".", epidemiology, min_graph_size=8)
ds

Processing...
100%|██████████| 14320/14320 [17:03<00:00, 13.99it/s]
Done!


ExampleDatasetClass(94300)

In [28]:
ds[100]

Data(x=[65, 1], edge_index=[2, 306], edge_attr=[306, 1], pos=[65, 2], new_epidemiology=[65, 3], cumulative_epidemiology=[65, 3], aggregation_level=1, window_final_day=2021-04-04, window_size=7, grouped_by_country='', component_index=1)

In [25]:
ds.get_summary()

100%|██████████| 94300/94300 [00:03<00:00, 23807.43it/s]


ExampleDatasetClass (#graphs=94300):
+------------+----------+----------+
|            |   #nodes |   #edges |
|------------+----------+----------|
| mean       |     41.2 |    143.6 |
| std        |     93.9 |    349.9 |
| min        |      8   |     14   |
| quantile25 |     10   |     28   |
| median     |     16   |     46   |
| quantile75 |     35   |    122   |
| max        |   2162   |   8112   |
+------------+----------+----------+

And it's done! A Torch Geometric dataset with over 94K entries and with medians of 16 nodes and 23 undirected edges (the database saves each undirected edge as 2 directed ones, so the number on the table is doubled). This dataset can now be incorporated into a DataLoader, transformed/pre-processed and used for GNN training and/or inference. We won't show it being actually used this way here, as it's not unique at all to our work, but we hope this example use case illustrated how to bridge any gap between C19EpidemiologyGraphs and third-party libraries.

In [29]:
ds.close()